# Visualbert v2 ONNX Model

In [3]:
from unitorch.cli import cached_path
from transformers import BertTokenizer

def get_bert_tokenizer(
    vocab_path,
    do_lower_case = True,
    do_basic_tokenize = True,
    special_input_ids = dict(),
):
    tokenizer = BertTokenizer(
        vocab_path,
        do_lower_case=do_lower_case,
        do_basic_tokenize=do_basic_tokenize,
    )
    for token, _id in special_input_ids.items():
        tokenizer.added_tokens_encoder[token] = _id
        tokenizer.unique_no_split_tokens.append(token)
        tokenizer.added_tokens_decoder[_id] = token
        tokenizer.add_tokens(token, special_tokens=True)
    return tokenizer

vocab_path = cached_path("https://huggingface.co/bert-base-uncased/resolve/main/vocab.txt")
tokenizer = get_bert_tokenizer(vocab_path = vocab_path)
tokens = tokenizer.tokenize("clothing shoes jewelry women clothing coats jackets vests")
tokens = [tokenizer.cls_token] + tokens + [tokenizer.sep_token]
print(tokens)
input_ids = tokenizer.convert_tokens_to_ids(tokens)
print(input_ids)

['[CLS]', 'clothing', 'shoes', 'jewelry', 'women', 'clothing', 'coats', 'jackets', 'vest', '##s', '[SEP]']
[101, 5929, 6007, 11912, 2308, 5929, 15695, 17764, 17447, 2015, 102]


In [2]:
from unitorch_microsoft.vpr.processing import VisualBertProcessor
from unitorch.cli import cached_path

processor = VisualBertProcessor(
    vocab_path=cached_path(
        "https://huggingface.co/bert-base-uncased/resolve/main/vocab.txt"
    ),
    max_seq_length=36,
)
tokens = processor._classification(
    "clothing shoes jewelry women clothing coats jackets vests"
)
print(tokens.__tensors__)

{'input_ids': tensor([  101,  5929,  6007, 11912,  2308,  5929, 15695, 17764, 17447,  2015,
          102,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0]), 'attention_mask': tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]), 'token_type_ids': tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]), 'position_ids': tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
        18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35])}


In [1]:
import torch
import torch.nn as nn
from unitorch.cli import cached_path

from transformers.models.visual_bert.modeling_visual_bert import (
    VisualBertConfig,
    VisualBertModel,
)

class VisualBertV2ForQuery(nn.Module):
    def __init__(
        self,
        config_path,
        image_embed_dim = 100,
        projection_dim = 100,
        gradient_checkpointing = False,
        weight_path = None
    ):
        super().__init__()
        self.config = VisualBertConfig.from_json_file(config_path)
        self.config.gradient_checkpointing = gradient_checkpointing
        self.config.visual_embedding_dim = self.config.hidden_size
        self.image_embed_dim = image_embed_dim

        self.image_conv = nn.Linear(image_embed_dim, self.config.hidden_size)
        self.query_bert = VisualBertModel(self.config, add_pooling_layer=False)
        self.projection_dim = projection_dim
        self.query_embed_dim = self.config.hidden_size
        self.query_projection = nn.Linear(self.query_embed_dim, self.projection_dim)

        self.from_checkpoint(weight_path)
    
    def from_checkpoint(self, weight_path):
        state_dict = torch.load(weight_path, map_location="cpu")
        self.load_state_dict(state_dict, strict=False)

    def forward(
        self,
        query_input_ids=None,
        query_image_embeds=None,
    ):
        query_input_ids = query_input_ids.reshape(1, -1)
        query_input_ids[query_input_ids > 30521] = 30521
        query_input_ids[query_input_ids < 0] = 0

        query_image_embeds = query_image_embeds.reshape(1, 1, 100)
        query_image_embeds = self.image_conv(query_image_embeds)
        query_outputs = self.query_bert(
            query_input_ids,
            attention_mask=None,
            token_type_ids=None,
            position_ids=None,
            visual_embeds=query_image_embeds,
            visual_attention_mask=None,
            visual_token_type_ids=None,
        )
        query_embeds = self.query_projection(query_outputs[0][:, 0])
        query_embeds = query_embeds / query_embeds.norm(dim=-1, keepdim=True)
        
        return query_embeds.view(1,100), query_embeds[:, 0].view(1).squeeze()


model_file = "/disk/lixin/ckpt/visualbert/pytorch_model.bin"
model = VisualBertV2ForQuery(
    config_path=cached_path("./configs/visualbert.argus.json"), weight_path=model_file
)
model.eval()

2023-08-14 10:48:29 AM : WARNING : detectron2 model can't be imported.
2023-08-14 10:48:29 AM : WARNING : fairseq model can't be imported.
2023-08-14 10:48:29 AM : WARNING : vlp model can't be imported.
2023-08-14 10:48:29 AM : WARNING : segformer model can't be imported.
2023-08-14 10:48:30 AM : WARNING : detr model can't be imported.
2023-08-14 10:48:30 AM : WARNING : detectron2 cli model can't be imported.
2023-08-14 10:48:30 AM : WARNING : fairseq cli model can't be imported.
2023-08-14 10:48:30 AM : WARNING : vlp cli model can't be imported.
2023-08-14 10:48:30 AM : WARNING : segformer cli model can't be imported.
2023-08-14 10:48:30 AM : WARNING : detr cli model can't be imported.


VisualBertV2ForQuery(
  (image_conv): Linear(in_features=100, out_features=768, bias=True)
  (query_bert): VisualBertModel(
    (embeddings): VisualBertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.0, inplace=False)
      (visual_token_type_embeddings): Embedding(2, 768)
      (visual_position_embeddings): Embedding(512, 768)
      (visual_projection): Linear(in_features=768, out_features=768, bias=True)
    )
    (encoder): VisualBertEncoder(
      (layer): ModuleList(
        (0-3): 4 x VisualBertLayer(
          (attention): VisualBertAttention(
            (self): VisualBertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value):

In [2]:
query = torch.tensor(
    [101, 5929, 6007, 11912, 2308, 5929, 15695, 17764, 17447, 2015, 102]
).unsqueeze(-1)
img_embed = "-0.04616177 -0.1159673 -0.08528523 0.1905098 0.03709327 -0.01312277 0.06673015 -0.08224729 -0.01040809 0.04598344 -0.1462961 -0.08063745 0.0884358 -0.01628013 0.148392 0.1082236 -0.1765878 0.09306743 0.0122325 -0.08560283 -0.05029535 0.03084217 0.01596232 -0.1859483 -0.002774475 0.07493474 -0.1663142 0.05123898 0.09619494 -0.1338701 0.08091179 -0.0679644 -0.03450328 0.1885101 0.2552213 0.1810764 -0.06981172 -0.02060077 0.09823886 0.03618161 -0.02384998 0.05091225 0.1621212 -0.01318149 -0.01524111 -0.08737237 -0.08761846 -0.1222196 0.1103951 0.06298082 0.05506718 0.04463443 -0.1053156 0.05489326 0.1009745 -0.06218047 -0.2151767 0.2009955 0.07995496 -0.1002232 -0.03015113 0.01651702 -0.0504215 0.05088477 -0.1110381 -0.08662394 0.24704 -0.06810931 -0.142163 0.02408212 -0.06323168 -0.1578443 0.04513462 -0.05154346 -0.04670211 -0.1431078 -0.06747935 0.07811068 0.09118298 -0.0676966 -0.001526211 -0.01333729 0.08748315 -0.1061855 -0.005310548 -0.03976621 -0.03471386 -0.06432562 -0.09493478 -0.1295277 -0.04983081 -0.00206829 0.1023707 0.02492245 0.203907 -0.09249474 -0.01149014 -0.003467969 -0.1368538 -0.07677495"
img_embed_list = list(map(float, img_embed.split(" ")))
image_embeds = torch.tensor(img_embed_list).unsqueeze(0)

torch.set_printoptions(profile="full")
torch.onnx.export(model, (query, image_embeds), 
    f = './visualbertv2.onnx',
    input_names = ['query', 'AVEV9Str'],
    output_names = ['qvec', 'dummy_score'],
    export_params = True,
    dynamic_axes = {'query' : {0: 'querylen'}},
    do_constant_folding=True,
    verbose=False,
    opset_version=13,
)

============= Diagnostic Run torch.onnx.export version 2.0.1+cu117 =============
verbose: False, log level: Level.ERROR
======================= 0 NONE 0 NOTE 0 WARNING 0 ERROR ========================



In [3]:
import torch
import onnxruntime as ort

query = torch.tensor(
    [101, 5929, 6007, 11912, 2308, 5929, 15695, 17764, 17447, 2015, 102]
).unsqueeze(-1)
img_embed = "-0.04616177 -0.1159673 -0.08528523 0.1905098 0.03709327 -0.01312277 0.06673015 -0.08224729 -0.01040809 0.04598344 -0.1462961 -0.08063745 0.0884358 -0.01628013 0.148392 0.1082236 -0.1765878 0.09306743 0.0122325 -0.08560283 -0.05029535 0.03084217 0.01596232 -0.1859483 -0.002774475 0.07493474 -0.1663142 0.05123898 0.09619494 -0.1338701 0.08091179 -0.0679644 -0.03450328 0.1885101 0.2552213 0.1810764 -0.06981172 -0.02060077 0.09823886 0.03618161 -0.02384998 0.05091225 0.1621212 -0.01318149 -0.01524111 -0.08737237 -0.08761846 -0.1222196 0.1103951 0.06298082 0.05506718 0.04463443 -0.1053156 0.05489326 0.1009745 -0.06218047 -0.2151767 0.2009955 0.07995496 -0.1002232 -0.03015113 0.01651702 -0.0504215 0.05088477 -0.1110381 -0.08662394 0.24704 -0.06810931 -0.142163 0.02408212 -0.06323168 -0.1578443 0.04513462 -0.05154346 -0.04670211 -0.1431078 -0.06747935 0.07811068 0.09118298 -0.0676966 -0.001526211 -0.01333729 0.08748315 -0.1061855 -0.005310548 -0.03976621 -0.03471386 -0.06432562 -0.09493478 -0.1295277 -0.04983081 -0.00206829 0.1023707 0.02492245 0.203907 -0.09249474 -0.01149014 -0.003467969 -0.1368538 -0.07677495"
img_embed_list = list(map(float, img_embed.split(" ")))
image_embeds = torch.tensor(img_embed_list).unsqueeze(0)

ort_session = ort.InferenceSession("./visualbertv2.onnx")
outputs = ort_session.run(
    None,
    {"query": query.cpu().numpy(), "AVEV9Str": image_embeds.cpu().numpy()},
)
print('onnx output: ', outputs)
embedding, dummy_score = model(query, image_embeds)
print(embedding)
print(dummy_score)


onnx output:  [array([[-0.06081162, -0.03298331, -0.02818314, -0.01249566, -0.00582739,
        -0.25797445, -0.10403566,  0.01745681, -0.23455863,  0.10266665,
         0.12318093, -0.04647967, -0.0072476 ,  0.20286177, -0.00250073,
         0.05387575, -0.07634564,  0.04856525, -0.00790471, -0.02231907,
         0.00295444,  0.09609099, -0.04772776,  0.05741395,  0.04241585,
         0.10079138,  0.07543562, -0.08022112, -0.12444132, -0.17846222,
         0.06484692,  0.06744181, -0.0642443 ,  0.06879411,  0.15046994,
         0.11157829, -0.02037061,  0.04459621,  0.17621976,  0.00550448,
        -0.07507644,  0.00573008, -0.18123809, -0.04104552,  0.1343572 ,
         0.09438052, -0.11466891,  0.0523861 ,  0.04391489, -0.23546296,
         0.04296859,  0.0743733 , -0.01072093, -0.14011459, -0.04310161,
         0.1538595 , -0.18232156, -0.08572671, -0.05592819,  0.06338545,
         0.19374466,  0.00230498,  0.00585019, -0.06948064,  0.13878147,
        -0.08073088, -0.07057615,  0

In [5]:
import torch
import onnxruntime as ort

query = torch.tensor([101, 8145, 7028, 2015, 2986, 2396, 2998, 7008, 102]).unsqueeze(-1)
img_embed = "0.06872461 0.06707684 -0.1004234 0.157478 0.005788147 0.05468951 -0.1689698 -0.2161552 0.005107108 -0.1413067 -0.2213147 -0.02213422 0.09381612 0.01865214 0.04586374 -0.0002564177 0.03646244 0.05179495 -0.05045513 0.03800458 -0.1236463 -0.09200428 -0.008652093 0.1573422 0.1308466 0.04061916 0.04233519 0.126397 -0.1020023 0.01449885 -0.03855991 0.009280446 -0.02262367 -0.02973268 0.0554627 -0.101397 0.2015726 0.01195515 0.04804528 0.1362682 0.08004167 -0.1898625 0.00134357 -0.04880193 0.1572406 -0.1457732 -0.06192927 -0.0002384908 -0.04287158 -0.1308655 0.0839516 0.08932887 -0.02052006 -0.06518085 -0.02623975 -0.1054462 0.06900226 0.1614499 0.1670368 -0.2113939 0.06217249 0.01754874 -0.1132064 -0.01737586 0.00583668 0.0410508 0.02069465 0.03145327 -0.09015376 0.05931452 -0.06863946 0.04490919 -0.2106095 0.1168717 -0.04938331 0.00816984 0.07793544 0.2658825 -0.03320502 0.1367838 -0.00196413 -0.000946969 -0.05130554 -0.1619273 -0.0199189 0.1331647 -0.04231874 0.2170963 0.1372149 0.01060467 0.1185931 0.0985478 0.02126219 -0.09441158 0.05213621 -0.0914111 -0.06110685 0.0133629 -0.04279166 0.07691804"
img_embed_list = list(map(float, img_embed.split(" ")))
image_embeds = torch.tensor(img_embed_list).unsqueeze(0)

ort_session = ort.InferenceSession("./visualbertv2.onnx")
outputs = ort_session.run(
    None,
    {"query": query.cpu().numpy(), "AVEV9Str": image_embeds.cpu().numpy()},
)
print('onnx output: ', outputs)
embedding, dummy_score = model(query, image_embeds)
print(embedding)
print(dummy_score)


onnx output:  [array([[-0.142448  , -0.04799994,  0.05782175,  0.09107915, -0.07493806,
        -0.25239116,  0.0044357 ,  0.05559996, -0.18184046,  0.08379126,
         0.12601972,  0.04278481, -0.0718069 ,  0.19978833, -0.05061267,
         0.09094395,  0.04122613,  0.00435682, -0.02002704,  0.00535572,
         0.13303727,  0.05332153,  0.00090446,  0.08511774, -0.07398861,
         0.13552184,  0.02432168, -0.05885392, -0.06546005, -0.16981907,
         0.03068335,  0.00419764, -0.0633379 ,  0.06464934,  0.12548496,
         0.11570881,  0.07756067,  0.12392344,  0.05487761,  0.03257694,
        -0.17666931, -0.02665   , -0.16078913,  0.00936321,  0.11802353,
         0.12528867, -0.11465858,  0.14629807,  0.0893579 , -0.23547548,
        -0.05288284,  0.04559559,  0.06468255, -0.13812834,  0.02677324,
         0.084852  , -0.12005285,  0.03430833,  0.02065865, -0.06146957,
         0.15377039,  0.00435042,  0.05172711, -0.08834646,  0.10408767,
        -0.00674029, -0.01380108,  0

In [6]:
for x in ort_session.get_inputs():
    print(x)
for x in ort_session.get_outputs():
    print(x)

NodeArg(name='query', type='tensor(int64)', shape=['querylen', 1])
NodeArg(name='AVEV9Str', type='tensor(float)', shape=[1, 100])
NodeArg(name='qvec', type='tensor(float)', shape=[1, 100])
NodeArg(name='dummy_score', type='tensor(float)', shape=[])


In [6]:
import onnxruntime.quantization as quantization

quantization.quantize_dynamic(
    "/disk/lixin/ckpt/visualbert/visualbertv2.onnx",
    "/disk/lixin/ckpt/visualbert/visualbertv2_int8.onnx",
)

2023-08-14 11:04:54 AM : INFO : Quantization parameters for tensor:"/Reshape_1_output_0" not specified
2023-08-14 11:04:54 AM : INFO : Quantization parameters for tensor:"/image_conv/Add_output_0" not specified
2023-08-14 11:04:56 AM : INFO : Quantization parameters for tensor:"/query_bert/embeddings/LayerNorm/Add_1_output_0" not specified
2023-08-14 11:04:56 AM : INFO : Quantization parameters for tensor:"/query_bert/encoder/layer.0/attention/self/Reshape_3_output_0" not specified
2023-08-14 11:04:56 AM : INFO : Quantization parameters for tensor:"/query_bert/encoder/layer.0/attention/output/LayerNorm/Add_1_output_0" not specified


Ignore MatMul due to non constant B: /[/query_bert/encoder/layer.0/attention/self/MatMul]
Ignore MatMul due to non constant B: /[/query_bert/encoder/layer.0/attention/self/MatMul_1]


2023-08-14 11:04:57 AM : INFO : Quantization parameters for tensor:"/query_bert/encoder/layer.0/intermediate/intermediate_act_fn/Mul_1_output_0" not specified
2023-08-14 11:04:57 AM : INFO : Quantization parameters for tensor:"/query_bert/encoder/layer.0/output/LayerNorm/Add_1_output_0" not specified
2023-08-14 11:04:57 AM : INFO : Quantization parameters for tensor:"/query_bert/encoder/layer.1/attention/self/Reshape_3_output_0" not specified


Ignore MatMul due to non constant B: /[/query_bert/encoder/layer.1/attention/self/MatMul]
Ignore MatMul due to non constant B: /[/query_bert/encoder/layer.1/attention/self/MatMul_1]


2023-08-14 11:04:58 AM : INFO : Quantization parameters for tensor:"/query_bert/encoder/layer.1/attention/output/LayerNorm/Add_1_output_0" not specified
2023-08-14 11:04:58 AM : INFO : Quantization parameters for tensor:"/query_bert/encoder/layer.1/intermediate/intermediate_act_fn/Mul_1_output_0" not specified
2023-08-14 11:04:59 AM : INFO : Quantization parameters for tensor:"/query_bert/encoder/layer.1/output/LayerNorm/Add_1_output_0" not specified
2023-08-14 11:04:59 AM : INFO : Quantization parameters for tensor:"/query_bert/encoder/layer.2/attention/self/Reshape_3_output_0" not specified


Ignore MatMul due to non constant B: /[/query_bert/encoder/layer.2/attention/self/MatMul]
Ignore MatMul due to non constant B: /[/query_bert/encoder/layer.2/attention/self/MatMul_1]


2023-08-14 11:04:59 AM : INFO : Quantization parameters for tensor:"/query_bert/encoder/layer.2/attention/output/LayerNorm/Add_1_output_0" not specified
2023-08-14 11:04:59 AM : INFO : Quantization parameters for tensor:"/query_bert/encoder/layer.2/intermediate/intermediate_act_fn/Mul_1_output_0" not specified
2023-08-14 11:05:00 AM : INFO : Quantization parameters for tensor:"/query_bert/encoder/layer.2/output/LayerNorm/Add_1_output_0" not specified
2023-08-14 11:05:00 AM : INFO : Quantization parameters for tensor:"/query_bert/encoder/layer.3/attention/self/Reshape_3_output_0" not specified


Ignore MatMul due to non constant B: /[/query_bert/encoder/layer.3/attention/self/MatMul]
Ignore MatMul due to non constant B: /[/query_bert/encoder/layer.3/attention/self/MatMul_1]


2023-08-14 11:05:00 AM : INFO : Quantization parameters for tensor:"/query_bert/encoder/layer.3/attention/output/LayerNorm/Add_1_output_0" not specified
2023-08-14 11:05:01 AM : INFO : Quantization parameters for tensor:"/query_bert/encoder/layer.3/intermediate/intermediate_act_fn/Mul_1_output_0" not specified
2023-08-14 11:05:01 AM : INFO : Quantization parameters for tensor:"/Gather_output_0" not specified


In [9]:
query = torch.tensor([101, 8145, 7028, 2015, 2986, 2396, 2998, 7008, 102]).unsqueeze(-1)
img_embed = "0.06872461 0.06707684 -0.1004234 0.157478 0.005788147 0.05468951 -0.1689698 -0.2161552 0.005107108 -0.1413067 -0.2213147 -0.02213422 0.09381612 0.01865214 0.04586374 -0.0002564177 0.03646244 0.05179495 -0.05045513 0.03800458 -0.1236463 -0.09200428 -0.008652093 0.1573422 0.1308466 0.04061916 0.04233519 0.126397 -0.1020023 0.01449885 -0.03855991 0.009280446 -0.02262367 -0.02973268 0.0554627 -0.101397 0.2015726 0.01195515 0.04804528 0.1362682 0.08004167 -0.1898625 0.00134357 -0.04880193 0.1572406 -0.1457732 -0.06192927 -0.0002384908 -0.04287158 -0.1308655 0.0839516 0.08932887 -0.02052006 -0.06518085 -0.02623975 -0.1054462 0.06900226 0.1614499 0.1670368 -0.2113939 0.06217249 0.01754874 -0.1132064 -0.01737586 0.00583668 0.0410508 0.02069465 0.03145327 -0.09015376 0.05931452 -0.06863946 0.04490919 -0.2106095 0.1168717 -0.04938331 0.00816984 0.07793544 0.2658825 -0.03320502 0.1367838 -0.00196413 -0.000946969 -0.05130554 -0.1619273 -0.0199189 0.1331647 -0.04231874 0.2170963 0.1372149 0.01060467 0.1185931 0.0985478 0.02126219 -0.09441158 0.05213621 -0.0914111 -0.06110685 0.0133629 -0.04279166 0.07691804"
img_embed_list = list(map(float, img_embed.split(" ")))
image_embeds = torch.tensor(img_embed_list).unsqueeze(0)

ort_session = ort.InferenceSession("/disk/lixin/ckpt/visualbert/visualbertv2.onnx")
outputs = ort_session.run(
    None,
    {"query": query.cpu().numpy(), "AVEV9Str": image_embeds.cpu().numpy()},
)
print("onnx output: ", outputs)

ort_session = ort.InferenceSession("/disk/lixin/ckpt/visualbert/visualbertv2_int8.onnx")
outputs = ort_session.run(
    None,
    {"query": query.cpu().numpy(), "AVEV9Str": image_embeds.cpu().numpy()},
)
print("onnx output: ", outputs)

onnx output:  [array([[-0.142448  , -0.04799994,  0.05782175,  0.09107915, -0.07493806,
        -0.25239116,  0.0044357 ,  0.05559996, -0.18184046,  0.08379126,
         0.12601972,  0.04278481, -0.0718069 ,  0.19978833, -0.05061267,
         0.09094395,  0.04122613,  0.00435682, -0.02002704,  0.00535572,
         0.13303727,  0.05332153,  0.00090446,  0.08511774, -0.07398861,
         0.13552184,  0.02432168, -0.05885392, -0.06546005, -0.16981907,
         0.03068335,  0.00419764, -0.0633379 ,  0.06464934,  0.12548496,
         0.11570881,  0.07756067,  0.12392344,  0.05487761,  0.03257694,
        -0.17666931, -0.02665   , -0.16078913,  0.00936321,  0.11802353,
         0.12528867, -0.11465858,  0.14629807,  0.0893579 , -0.23547548,
        -0.05288284,  0.04559559,  0.06468255, -0.13812834,  0.02677324,
         0.084852  , -0.12005285,  0.03430833,  0.02065865, -0.06146957,
         0.15377039,  0.00435042,  0.05172711, -0.08834646,  0.10408767,
        -0.00674029, -0.01380108,  0